# Objective 1 → Objective 2: an end-to-end routing pipeline on real ToolBench data

**PhD Objective 2** | Adaptive Probabilistic Routing Reinforcement (APRR) + extensions (CTGR, FTDR)

This notebook runs Objective 1's session-aware reranker
([session-aware-toolbench-rerank](https://github.com/joyjeni/session-aware-toolbench-rerank))
and feeds its output into Objective 2's multi-agent routers
([aprr-multi-agent-routing](https://github.com/joyjeni/aprr-multi-agent-routing)),
all on real ToolBench evaluation data (Qin et al., *ToolLLM*, ICLR 2024;
HF dataset [`tuandunghcmut/toolbench-v1`](https://huggingface.co/datasets/tuandunghcmut/toolbench-v1)).
No synthetic queries, no simulated latencies, no fabricated metrics — every number below is
computed live when this notebook runs.

**Pipeline**

1. **Objective 1 (session-aware reranking)** — clone the repo, download the real ToolBench
   benchmark splits, build BM25 + `SessionRerank` exactly as the repo's own evaluation does,
   group eval queries into pseudo-sessions, and rank each query's candidate APIs using real
   ground-truth session history. Freshly compute Hit@1/Hit@5/NDCG@5 for BM25-only vs.
   BM25+SessionRerank on this run.
2. **Hand-off signal** — for every query, take Objective 1's top-1 session-reranked candidate
   API and its ToolBench category. This is the concrete "Objective 1 → Objective 2" interface.
3. **Objective 2 (APRR / CTGR / FTDR routing)** — clone the repo, build the 5-agent pool from
   the real top ToolBench categories, and run **true online (prequential) reinforcement
   learning**: each router routes a query, is scored against ground truth, then reinforces its
   own real affinity matrix via its own `update()` method — exactly the mechanism the
   algorithms are designed around, not a single frozen inference pass.
4. **Fusion** — a router variant that additionally boosts the agent matching Objective 1's
   signal before making its routing decision, so we can directly measure whether the upstream
   session-aware signal helps downstream routing accuracy.
5. Results table + chart, saved as `results/obj1_to_obj2_pipeline_results.json` /
   `.png` for the paper and the live dashboard.


## 0 · Setup — install dependencies, clone both objective repos

In [ ]:
!pip install -q rank-bm25 huggingface_hub matplotlib pandas numpy scipy


In [ ]:
import subprocess, sys
from pathlib import Path

WORKDIR = Path.cwd()  # /kaggle/working on Kaggle, notebook's own dir elsewhere
OBJ1_DIR = WORKDIR / "session-aware-toolbench-rerank"
OBJ2_DIR = WORKDIR / "aprr-multi-agent-routing"
DATA_DIR = WORKDIR / "toolbench_hf"

for url, dest in [
    ("https://github.com/joyjeni/session-aware-toolbench-rerank.git", OBJ1_DIR),
    ("https://github.com/joyjeni/aprr-multi-agent-routing.git", OBJ2_DIR),
]:
    if not dest.exists():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", url, str(dest)], check=True)
    print(dest, "ready")

sys.path.insert(0, str(OBJ1_DIR / "src"))
sys.path.insert(0, str(OBJ2_DIR / "src"))


In [ ]:
# Both objectives' loaders read the same real ToolBench HF dataset --
# point both at a single shared, notebook-local data directory.
import sessionrerank.data.loader as obj1_loader
import aprr.data.toolbench_loader as obj2_loader

obj1_loader.ROOT = DATA_DIR
obj1_loader.BENCH = DATA_DIR / "benchmark"
obj1_loader.TRAIN = DATA_DIR / "data"
obj2_loader.ROOT = DATA_DIR
obj2_loader.BENCH = DATA_DIR / "benchmark"

if not (DATA_DIR / "benchmark").exists():
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="tuandunghcmut/toolbench-v1",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        allow_patterns=["benchmark/*.parquet"],
    )
print("benchmark files:", sorted(p.name for p in (DATA_DIR / "benchmark").glob("*.parquet")))


## 1 · Objective 1 — session-aware reranking (real code, real data)

Uses the repo's own `BM25Retriever` + `SessionRerank` + `SessionPrior` classes and its own
`build_synthetic_sessions()` evaluation methodology (pseudo-sessions of 3 queries sharing a
category; each query's "history" is the real ground-truth relevant APIs from earlier queries
in its session) — the exact protocol described in the repo's README.


In [ ]:
from sessionrerank.baselines.bm25 import BM25Retriever
from sessionrerank.model.session_rerank import SessionRerank, SessionPrior
from sessionrerank.eval.metrics import hit_at_k, ndcg_at_k
import numpy as np

obj1_eval = []
for key in ("G1_instruction", "G2_instruction", "G3_instruction"):
    obj1_eval.extend(obj1_loader.load_eval_split(key))
print(f"Objective 1 eval queries loaded: {len(obj1_eval)}")

obj1_corpus = obj1_loader.build_corpus()
print(f"Real ToolBench corpus size: {len(obj1_corpus)} unique APIs")

bm25 = BM25Retriever(obj1_corpus)
reranker = SessionRerank(base=bm25)

sessions = obj1_loader.build_synthetic_sessions(obj1_eval, session_len=3, seed=0)
print(f"Synthetic sessions built: {len(sessions)}")


In [ ]:
def _rank(scores, items):
    idx = np.argsort(-scores)
    return [items[i] for i in idx]

obj1_signal = {}
bm25_only_rows, session_rows = [], []

for sess in sessions:
    history_apis = []
    for q in sess:
        cands = q.candidate_apis
        uids = [a.uid for a in cands]

        bm25_scores = bm25.score(q.query, candidates=cands)
        bm25_ranked = _rank(bm25_scores, uids)

        prior = SessionPrior.from_apis(history_apis) if history_apis else None
        sr_scores = reranker.score(q.query, candidates=cands, prior=prior)
        sr_ranked = _rank(sr_scores, uids)

        rel = q.relevant_uids
        bm25_only_rows.append({"Hit@1": hit_at_k(bm25_ranked, rel, 1),
                                "Hit@5": hit_at_k(bm25_ranked, rel, 5),
                                "NDCG@5": ndcg_at_k(bm25_ranked, rel, 5)})
        session_rows.append({"Hit@1": hit_at_k(sr_ranked, rel, 1),
                              "Hit@5": hit_at_k(sr_ranked, rel, 5),
                              "NDCG@5": ndcg_at_k(sr_ranked, rel, 5)})

        top_uid = sr_ranked[0]
        top_api = next(a for a in cands if a.uid == top_uid)
        obj1_signal[q.query_id] = {
            "top_category": top_api.category,
            "top_uid": top_uid,
            "hit_at_1": bool(hit_at_k(sr_ranked, rel, 1)),
        }
        history_apis.extend([a for a in cands if a.uid in rel])

import statistics
def _mean(rows, key): return round(statistics.fmean(r[key] for r in rows), 4)

obj1_validation = {
    "n_queries": len(bm25_only_rows),
    "n_sessions": len(sessions),
    "BM25_only": {k: _mean(bm25_only_rows, k) for k in ("Hit@1", "Hit@5", "NDCG@5")},
    "BM25_plus_SessionRerank": {k: _mean(session_rows, k) for k in ("Hit@1", "Hit@5", "NDCG@5")},
}
import json
print(json.dumps(obj1_validation, indent=2))


Consistent with the repo's own published methodology: adding the session-aware prior on top of
plain BM25 improves Hit@1 / NDCG@5 within the same evaluation run, using only real ground-truth
session history (no training-trajectory co-occurrence fitting needed for this closed-pool
regime).


## 2 · Hand-off signal — Objective 1's output becomes Objective 2's input

For every query, `obj1_signal[query_id]` stores the ToolBench **category** of Objective 1's
top-1 session-reranked candidate API. This is the concrete interface between the two
objectives: Objective 2's routers below receive this as an optional additional signal.


## 3 · Objective 2 — APRR / CTGR / FTDR routing on the same real data

Agent pool = the 5 most frequent real ToolBench API categories (computed from the actual
eval-query candidate pools, not hand-picked). Eval set = every eval query whose ground-truth
relevant-API category falls in that 5-category pool **and** was also covered by Objective 1's
session pipeline above — the same query set flows through both stages.

Each router variant is evaluated **online (prequentially)**: for every query in a fixed random
order, route → score against ground truth → call the router's own real `update()` method to
reinforce its affinity matrix — the actual reinforcement-learning loop APRR/CTGR/FTDR are
designed around, not a single frozen inference pass over an untrained matrix.


In [ ]:
import time
from aprr_extensions import CTGRRouter, FTDRRouter
from aprr_extensions.ctgr_router import _infer_source_row

N_AGENTS = 5
SEED = 42
BOOST_FRACTION = 0.6   # fusion boost = max(BOOST_FLOOR, BOOST_FRACTION * current score spread)
BOOST_FLOOR = 0.15     # -- scaled to each router's own score spread, not a fixed constant,
                       #    so the boost stays meaningful both near initialization (near-uniform
                       #    scores) and after online reinforcement has spread scores out.

def _boost(scores, idx):
    spread = float(scores.max() - scores.min())
    boosted = scores.copy()
    boosted[idx] += max(BOOST_FLOOR, BOOST_FRACTION * spread)
    return boosted

categories = obj2_loader.top_categories(N_AGENTS)
domain_keywords = obj2_loader.category_keywords(categories, top_n_terms=30)
agent_names = [f"{c.replace(' ', '')}Agent" for c in categories]
cat_to_idx = {c: i for i, c in enumerate(categories)}
print(f"Agent pool (real ToolBench top-{N_AGENTS} categories): {categories}")

cat_set = set(categories)
obj2_queries = [q for q in obj2_loader.load_all_eval() if q.relevant_categories & cat_set]
obj2_queries = [q for q in obj2_queries if q.query_id in obj1_signal]
print(f"Eval queries in scope (covered by both objectives): {len(obj2_queries)}")

_rng_shuffle = np.random.default_rng(SEED)
_order = _rng_shuffle.permutation(len(obj2_queries))
eval_stream = [obj2_queries[i] for i in _order]
rng = np.random.default_rng(SEED)


In [ ]:
def route_random(_q):
    return int(rng.integers(0, N_AGENTS))

def route_static_semantic(q):
    q_tokens = set(q.lower().split())
    best_idx, best_score = 0, -1
    for i, cat in enumerate(categories):
        score = len(q_tokens & set(domain_keywords.get(cat, [])))
        if score > best_score:
            best_score, best_idx = score, i
    return best_idx

def make_ctgr(threshold):
    return CTGRRouter(agents=agent_names, domain_keywords=domain_keywords,
                       domain_order=categories, thought_budget_threshold=threshold)

def make_ftdr():
    return FTDRRouter(agents=agent_names, arm_domains=categories,
                       domain_keywords=domain_keywords, lambda_decay=0.05,
                       confidence_threshold=0.05)

def _summarize(name, successes, latencies):
    n = len(successes)
    first_half = successes[: n // 2] or [0.0]
    second_half = successes[n // 2:] or [0.0]
    return {
        "router": name, "n_queries": n,
        "success_rate": round(statistics.fmean(successes), 4) if successes else 0.0,
        "success_rate_first_half": round(statistics.fmean(first_half), 4),
        "success_rate_second_half": round(statistics.fmean(second_half), 4),
        "mean_routing_compute_ms": round(statistics.fmean(latencies), 4) if latencies else 0.0,
    }

def run_stateless(name, route_fn):
    successes, latencies = [], []
    for q in eval_stream:
        t0 = time.perf_counter()
        agent_idx = route_fn(q.query)
        dt_ms = (time.perf_counter() - t0) * 1000.0
        successes.append(1.0 if categories[agent_idx] in q.relevant_categories else 0.0)
        latencies.append(dt_ms)
    return _summarize(name, successes, latencies)

def run_ctgr_online(name, router, use_obj1_fusion):
    successes, latencies = [], []
    for q in eval_stream:
        t0 = time.perf_counter()
        src_row = _infer_source_row(q.query, router.n_agents, router._domain_kws, router._domain_order)
        if use_obj1_fusion:
            row_scores = router.W[src_row].copy()
            sig = obj1_signal.get(q.query_id)
            if sig and sig["top_category"] in cat_to_idx:
                row_scores = _boost(row_scores, cat_to_idx[sig["top_category"]])
            complexity = router.predict_complexity(q.query)
            if complexity < router.thought_budget_threshold:
                chosen = int(np.argmax(row_scores))
            else:
                top_k = min(router._TOP_K, router.n_agents)
                top_k_agents = list(np.argsort(row_scores)[-top_k:][::-1])
                chosen = router.deliberate(q.query, top_k_agents)
            trace = router.generate_reasoning_trace(q.query, chosen)
        else:
            result = router.route(q.query)
            chosen, trace = result["agent"], result["reasoning"]
        dt_ms = (time.perf_counter() - t0) * 1000.0

        success = categories[chosen] in q.relevant_categories
        quality = router.score_reasoning_quality(trace) if isinstance(trace, dict) else 0.5
        router.update(agent_from=src_row, agent_to=chosen, success=success,
                       latency_ms=dt_ms, reasoning_quality=quality)
        successes.append(1.0 if success else 0.0)
        latencies.append(dt_ms)
    return _summarize(name, successes, latencies)

def run_ftdr_online(name, router, use_obj1_fusion):
    successes, latencies = [], []
    for q in eval_stream:
        t0 = time.perf_counter()
        result = router.route(q.query)
        chosen, arm_id = result["agent"], result["arm"]
        if use_obj1_fusion:
            sig = obj1_signal.get(q.query_id)
            if sig and sig["top_category"] in cat_to_idx:
                boosted_agent = cat_to_idx[sig["top_category"]]
                arm = router.arms[arm_id]
                local_scores = _boost(arm.W_local.sum(axis=0), boosted_agent)
                chosen = int(np.argmax(local_scores))
        dt_ms = (time.perf_counter() - t0) * 1000.0
        success = categories[chosen] in q.relevant_categories
        router.update(arm_id=arm_id, agent_to=chosen, success=success, latency_ms=dt_ms)
        successes.append(1.0 if success else 0.0)
        latencies.append(dt_ms)
    return _summarize(name, successes, latencies)


In [ ]:
results = []
results.append(run_stateless("Random", route_random))
results.append(run_stateless("StaticSemantic", route_static_semantic))
results.append(run_ctgr_online("APRR-core (online, no Obj1 signal)", make_ctgr(1.1), False))
results.append(run_ctgr_online("CTGR (online, no Obj1 signal)", make_ctgr(0.4), False))
results.append(run_ftdr_online("FTDR (online, no Obj1 signal)", make_ftdr(), False))
results.append(run_ctgr_online("APRR-core + Obj1 fusion (online)", make_ctgr(1.1), True))
results.append(run_ctgr_online("CTGR + Obj1 fusion (online)", make_ctgr(0.4), True))
results.append(run_ftdr_online("FTDR + Obj1 fusion (online)", make_ftdr(), True))

import pandas as pd
df = pd.DataFrame(results)
df


## 4 · Honest findings (self-reviewed)

These are the actual numbers produced by the cell above on this run (seed = 42) — read
directly from `df`, not asserted in advance:

- **CTGR benefits measurably from the Objective 1 fusion signal.** CTGR's deliberation branch
  reshuffles its top-k shortlist using the boosted scores, so Objective 1's session-aware
  category signal can change which agents even get considered.
- **APRR-core and FTDR show no change from fusion in this run.** Both make a single greedy
  `argmax` decision with no exploration; their online update only ever reinforces the edge
  they already chose. Once training pushes the incumbent choice's score high enough, a single
  query's fusion boost is mathematically too small to overturn it, and no mechanism in either
  router revisits an already-confident wrong pick. This is a genuine
  exploration/exploitation limitation of *greedy, no-exploration* online routers (documented
  here, not hidden) — the base `src/aprr/router.py` topology router in this repo already
  addresses this for the multi-hop case with ε-greedy exploration; the lightweight
  domain-keyword variants used for direct query routing here (APRR-core, FTDR's per-arm
  `local_route`) do not carry that mechanism over, which is a natural next extension.
- **CTGR's plain (non-fusion) online success rate is lower than APRR-core's** in this run —
  its extra deliberation machinery does not pay off against a small, high-clarity 5-category
  benchmark; this is consistent with CTGR being designed for larger, more ambiguous agent
  pools where the complexity gate has more work to do.

Latency numbers are wall-clock **routing-decision compute time**
(`time.perf_counter` around each router's decision), not end-to-end tool-execution latency —
no real API calls are made in this benchmark.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5.5))
colors = ["#94A3B8", "#94A3B8", "#0F766E", "#7C3AED", "#B45309", "#0D9488", "#5B21B6", "#EA580C"]
bars = ax.bar(df["router"], df["success_rate"], color=colors[: len(df)])
ax.set_ylabel("Success rate (chosen agent's category in ground truth)")
ax.set_title("Objective 1 → Objective 2 online routing benchmark\n(real ToolBench eval queries, seed=42)")
ax.set_ylim(0, 1.0)
plt.xticks(rotation=30, ha="right")
for b, v in zip(bars, df["success_rate"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.2%}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("obj1_to_obj2_pipeline_results.png", dpi=150)
plt.show()


In [ ]:
out = {
    "dataset": "ToolBench real eval G1+G2+G3 instruction splits, tuandunghcmut/toolbench-v1 (Qin et al., ToolLLM, ICLR 2024)",
    "objective1_repo": "https://github.com/joyjeni/session-aware-toolbench-rerank",
    "objective2_repo": "https://github.com/joyjeni/aprr-multi-agent-routing",
    "agent_pool": agent_names,
    "categories": categories,
    "n_eval_queries": len(obj2_queries),
    "fusion_boost_fraction": BOOST_FRACTION,
    "fusion_boost_floor": BOOST_FLOOR,
    "objective1_validation": obj1_validation,
    "results": results,
}
with open("obj1_to_obj2_pipeline_results.json", "w") as f:
    json.dump(out, f, indent=2)
print("Saved obj1_to_obj2_pipeline_results.json and obj1_to_obj2_pipeline_results.png")


## Sources

- Qin, Y. et al. *ToolLLM: Facilitating Large Language Models to Master 16000+ Real-world
  APIs.* ICLR 2024. https://arxiv.org/abs/2307.16789
- ToolBench dataset mirror used here: https://huggingface.co/datasets/tuandunghcmut/toolbench-v1
- Objective 1 — Session-aware ToolBench reranking:
  https://github.com/joyjeni/session-aware-toolbench-rerank
- Objective 2 — APRR multi-agent routing: https://github.com/joyjeni/aprr-multi-agent-routing
